In [ ]:
# CELL 1 — Setup: env, deps, GPU probe (~10 min, ~0 VRAM)
import os
os.environ["NCCL_P2P_DISABLE"] = "1"   # 2xT4 sin NVLink: DDP over PCIe
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["RWKV_MY_TESTING"] = "x070"
os.environ["WKV"] = "fla"

# ACTIVE installs: remove incompatible torchao, install rwkv + rwkv-fla + lightning + deps
!pip uninstall -y torchao
!pip install -q --no-cache-dir rwkv rwkv-fla einops jsonlines peft datasets triton huggingface_hub "lightning>=2.0.0"
!pip cache purge
!nvidia-smi  # live T4x2 probe


In [ ]:
# CELL 2 — Model config: 2.9B DEFAULT via BlinkDL (~5.5 GiB download, ~5.8 GB VRAM bf16)
import os, shutil
from huggingface_hub import hf_hub_download

MODEL_REPO = "BlinkDL/rwkv7-g1"
MODEL_FILE = "rwkv7-g1j-2.9b-20260831-ctx16384.pth"
N_LAYER, N_EMBD = 32, 2560

os.makedirs("/kaggle/working/models", exist_ok=True)
model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, local_dir="/kaggle/working/models")
print("Model downloaded to:", model_path)

# Ensure model_path is a regular file (not a symlink to ~/.cache)
if os.path.islink(model_path):
    target = os.path.realpath(model_path)
    os.remove(model_path)
    shutil.copy2(target, model_path)

# Purge HuggingFace cache to reclaim ~5.8 GB disk space
shutil.rmtree(os.path.expanduser("~/.cache/huggingface"), ignore_errors=True)
print("Purged HuggingFace cache to reclaim ~5.8 GB disk space.")


In [ ]:
# CELL 3 — Dataset prep + DATA STAGING: robust copy from /kaggle/input
import shutil, os, glob

input_base = '/kaggle/input'
print("Scanning /kaggle/input:", os.listdir(input_base) if os.path.exists(input_base) else "None")

dataset_dir = None
for root, dirs, files in os.walk(input_base):
    if 'rwkv_kateto_base_train.jsonl' in files:
        if os.path.basename(root) == 'data':
            dataset_dir = os.path.dirname(root)
        else:
            dataset_dir = root
        break

if not dataset_dir:
    dataset_dir = '/kaggle/input/kateto-sft'

print(f"Using dataset source directory: {dataset_dir}")
target_dir = '/kaggle/working/kateto-train'
os.makedirs(target_dir, exist_ok=True)

# Copy all files recursively from dataset source
for src in glob.glob(f"{dataset_dir}/**/*", recursive=True):
    if os.path.isfile(src):
        rel = os.path.relpath(src, dataset_dir)
        dst = os.path.join(target_dir, rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)

# Symlink /kaggle/working/out to kateto-train/out for unified paths
os.makedirs("/kaggle/working/out", exist_ok=True)
if not os.path.exists(f"{target_dir}/out"):
    os.symlink("/kaggle/working/out", f"{target_dir}/out")

# Verification
required = [
    'data/rwkv_kateto_base_train.jsonl',
    'data/rwkv_kateto_seco.jsonl',
    'data/rwkv_kateto_streamer.jsonl',
    'data/rwkv_kateto_jane.jsonl',
    'data/rwkv_kateto_doktor.jsonl',
    'data/rwkv_kateto_whisperer.jsonl',
    'data/debate_speech.jsonl',
    'data/orpo/anti_sycophancy_rioplatense.jsonl',
    'config/state_tuning.yaml',
    'config/pissa_2.3b.yaml',
    'config/orpo.yaml',
    'docs/orpo-spike.md',
    'RWKV-PEFT/train.py',
    'rwkv_pipeline/train_state_voice.sh',
]
missing = [f for f in required if not os.path.exists(f'{target_dir}/{f}')]
assert not missing, f'STAGING FAIL — faltan: {missing}'
print(f'STAGING SUCCESS: {len(required)} archivos y componentes verificados en {target_dir}')
for r in required:
    fp = f'{target_dir}/{r}'
    sz = os.path.getsize(fp) if os.path.isfile(fp) else 0
    print(f'  {r}: {sz:,} bytes')


In [ ]:
# CELL 4 — Packing: FORBIDDEN by default, no-pack jsonl
DATA_TYPE = "jsonl"
CTX_LEN = 1024
print(f"DATA_TYPE: {DATA_TYPE}, CTX_LEN: {CTX_LEN}")


In [ ]:
# CELL 5 — PiSSA/LoRA config: wired dispatch r32/niter4
PEFT, PISSA_R, SVD_NITER = "pissa", 32, 4
FALLBACK_PEFT, FALLBACK_CFG = "lora", '{"r":16, "lora_alpha":32, "lora_dropout":0.01}'
PISSA_CFG = '{"pissa_load":"", "pissa_init":"", "pissa_r":32, "svd_niter":4}'
TARGETS = "receptance,key,value,output"
print(f"PEFT: {PEFT}, PISSA_CFG: {PISSA_CFG}")


In [ ]:
# CELL 6 — SFT Capa 1: 1 epoch (~1.7h), micro-bsz 4, DDP 2xT4
import os, subprocess, sys, torch

base_pth = "/kaggle/working/out/rwkv_kateto_base/rwkv-0.pth"
if os.path.exists(base_pth) and os.path.getsize(base_pth) > 1000000000:
    print(f">>> Checkpoint Capa 1 ya presente: {base_pth} ({os.path.getsize(base_pth):,} bytes) — OMITIENDO entrenamiento de Capa 1 <<<")
else:
    num_gpus = torch.cuda.device_count()
    print(f"Detected {num_gpus} GPU(s)")
    assert num_gpus >= 1, "No GPU detected!"

    devices_arg = str(num_gpus)
    strategy_arg = "ddp" if num_gpus > 1 else "auto"
    print(f"Using devices: {devices_arg}, strategy: {strategy_arg}")

    cmd_pissa = [
        "python", "train.py",
        "--load_model", "/kaggle/working/models/rwkv7-g1j-2.9b-20260831-ctx16384.pth",
        "--proj_dir", "/kaggle/working/out/rwkv_kateto_base",
        "--data_file", "/kaggle/working/kateto-train/data/rwkv_kateto_base_train.jsonl",
        "--data_type", "jsonl",
        "--loss_mask", "qa",
        "--vocab_size", "65536",
        "--n_layer", "32",
        "--n_embd", "2560",
        "--ctx_len", "1024",
        "--micro_bsz", "4",
        "--epoch_count", "1",
        "--epoch_save", "1",
        "--lr_init", "2e-4",
        "--lr_final", "2e-5",
        "--accelerator", "gpu",
        "--devices", devices_arg,
        "--strategy", strategy_arg,
        "--precision", "bf16",
        "--grad_cp", "1",
        "--my_testing", "x070",
        "--op", "fla",
        "--peft", "pissa",
        "--peft_config", "{}",
        "--pissa_config", '{"pissa_load":"","pissa_init":"","pissa_r":32,"svd_niter":4}',
        "--merge", "1"
    ]

    cwd = "/kaggle/working/kateto-train/RWKV-PEFT"
    os.makedirs("/kaggle/working/out/rwkv_kateto_base", exist_ok=True)
    print("Iniciando entrenamiento Capa 1 PiSSA...")
    res = subprocess.run(cmd_pissa, cwd=cwd)

    if res.returncode != 0:
        print(">>> PiSSA falló o no soportado, ejecutando fallback LoRA-r16...")
        with open("/kaggle/working/out/rwkv_kateto_base/pissa_fallback_reason.txt", "w") as f:
            f.write(f"pissa exit code: {res.returncode}\n")
        cmd_lora = [
            "python", "train.py",
            "--load_model", "/kaggle/working/models/rwkv7-g1j-2.9b-20260831-ctx16384.pth",
            "--proj_dir", "/kaggle/working/out/rwkv_kateto_base",
            "--data_file", "/kaggle/working/kateto-train/data/rwkv_kateto_base_train.jsonl",
            "--data_type", "jsonl",
            "--loss_mask", "qa",
            "--vocab_size", "65536",
            "--n_layer", "32",
            "--n_embd", "2560",
            "--ctx_len", "1024",
            "--micro_bsz", "4",
            "--epoch_count", "1",
            "--epoch_save", "1",
            "--lr_init", "2e-4",
            "--lr_final", "2e-5",
            "--accelerator", "gpu",
            "--devices", devices_arg,
            "--strategy", strategy_arg,
            "--precision", "bf16",
            "--grad_cp", "1",
            "--my_testing", "x070",
            "--op", "fla",
            "--peft", "lora",
            "--peft_config", '{"r":16,"lora_alpha":32,"lora_dropout":0.01}',
            "--merge", "1"
        ]
        res_lora = subprocess.run(cmd_lora, cwd=cwd)
        assert res_lora.returncode == 0, f"LoRA training failed with code {res_lora.returncode}"

    print(">>> CAPA 1 COMPLETADA CON ÉXITO <<<")

    # Reclamar 5.8 GB del modelo base original ahora que el modelo fusionado rwkv-0.pth está listo
    raw_model = "/kaggle/working/models/rwkv7-g1j-2.9b-20260831-ctx16384.pth"
    if os.path.exists(raw_model):
        os.remove(raw_model)
        print(f"Espacio liberado: se eliminó el modelo base original ({raw_model}).")


In [ ]:
# CELL 7 — ORPO: CONDITIONAL, SFT-only default
import os
RESPIKE_VERDICT = os.environ.get("ORPO_RESPIKE", "NO-GO")
assert os.path.exists("/kaggle/working/kateto-train/docs/orpo-spike.md"), "missing spike doc"
if RESPIKE_VERDICT != "GO":
    print("ORPO SKIPPED: spike verdict is NO-GO (docs/orpo-spike.md) - SFT-only path continues.")
else:
    print("ORPO GO: re-spike GO")


In [ ]:
# CELL 8 — State tuning per voice: micro_bsz=1, grad_cp=1 compatible con 2.9B en T4
import subprocess, os

env = os.environ.copy()
env["N_LAYER"] = "32"
env["N_EMBD"] = "2560"
env["MICRO_BSZ"] = "1"
env["GRAD_CP"] = "1"

voices = ["seco", "jane", "whisperer", "streamer"]
for v in voices:
    print(f"\n=======================================================")
    print(f">>> Ejecutando State Tuning para voz: {v}")
    print(f"=======================================================")
    cmd = ["bash", "rwkv_pipeline/train_state_voice.sh", v]
    res = subprocess.run(cmd, cwd="/kaggle/working/kateto-train", env=env)
    if res.returncode != 0:
        print(f"Advertencia: state tuning para {v} terminó con código {res.returncode}")
    else:
        print(f"✓ State tuning para {v} completado.")


In [ ]:
# CELL 9 — Eval + export: JSON-valid eval + debate probe + checkpoint copy
import subprocess, os, glob, shutil

print(">>> Verificando resultados y ejecutando probes...")
subprocess.run(["python", "scripts/probe_debate_no_toolcall.py"], cwd="/kaggle/working/kateto-train")

export_dir = "/kaggle/working/kateto-ckpt"
os.makedirs(export_dir, exist_ok=True)

# Exportar checkpoints base mediante symlink o copia
for pth in glob.glob("/kaggle/working/out/rwkv_kateto_base/*.pth"):
    dst = os.path.join(export_dir, os.path.basename(pth))
    if not os.path.exists(dst):
        try:
            os.symlink(pth, dst)
        except OSError:
            shutil.copy2(pth, dst)
        print(f"Export checkpoint base: {pth} -> {dst}")

# Copiar estados (archivos livianos de pocos MB)
states_src = "/kaggle/working/kateto-train/out/rwkv_states" if os.path.exists("/kaggle/working/kateto-train/out/rwkv_states") else "/kaggle/working/out/rwkv_states"
if os.path.exists(states_src):
    shutil.copytree(states_src, os.path.join(export_dir, "rwkv_states"), dirs_exist_ok=True)
    shutil.copytree(states_src, "/kaggle/working/rwkv_states", dirs_exist_ok=True)
    print(f"Estados exportados desde {states_src} a {export_dir}/rwkv_states")

print(f"\n>>> EXPORT COMPLETADO en {export_dir} <<<")
if os.path.exists(export_dir):
    print(os.listdir(export_dir))
